# Day 21 · OpenStreetMap — Does the Map Know the Curb?

**Question.** Wheelchair route planners built on OpenStreetMap (OSM) can only send a rider across a street
where the map says the curb at each end is lowered or flush. Chicago's sidewalks and crosswalks are now
drawn in OSM as lines of their own. At how many crosswalks does the map give the curb's height?

**Approach.** Every crosswalk way (`footway=crossing`) inside the city limits, from a dated Geofabrik
extract. A crosswalk end *gives the height* when the curb node at that end carries
`kerb=lowered|flush|raised|no`; `kerb=yes`, `kerb=unspecified` or a bare `barrier=kerb` mark a curb
without its height. Chicago's mappers put the curb node on the crosswalk itself, one of its first or last
two nodes (checked in QA). Each crosswalk counts as both ends, one end or neither.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Cut the city from the Illinois extract with `osmium`; read crosswalks, curbs, sidewalks, streets |
| 3. Validate | Tag values, counts |
| 4. Transform | End states; crosswalks in the city, by side and by station |
| 5. Analyze | Shares; the inset's two stations |
| 6. QA | Curb nodes off the crosswalk; the loose reading; when the lines were drawn |
| 7. Export | `plate.json` |

Run top to bottom from this folder. The crosswalks and their end values are committed as
`data/raw/osm_crosswalks_<extract date>.csv.gz`; the sidewalk and street lines are read from the extract
(`reference/raw/osm/`, gitignored) and cached in `data/processed/`.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import re
import subprocess
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely

from atlas import Ledger, write_plate
from atlas.grid import miles_from_origin
from atlas.walk import latest_pbf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day21')
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = cut the extract again

DAY = 21
KNOWN = ['lowered', 'flush', 'raised', 'no']           # kerb=* values that give the height
PASSABLE = ['lowered', 'flush', 'no']                  # a wheelchair can roll over these
QUARTER = 0.25                                         # miles around a station
INSET_RY = 0.36                                        # miles: the inset windows run a quarter mile east and west, this far north and south
MIN_CROSSWALKS = 40                                    # an inset station has at least this many in its quarter mile
SIDEWALK_NEAR_M = 30                                   # a street has a sidewalk if one is drawn this close to its midpoint
OSM_URL = 'https://www.openstreetmap.org/copyright'
GEOFABRIK_URL = 'https://download.geofabrik.de/north-america/us/illinois.html'
KERB_WIKI = 'https://wiki.openstreetmap.org/wiki/Key:kerb'
CROSSING_WIKI = 'https://wiki.openstreetmap.org/wiki/Tag:footway%3Dcrossing'
STREETS = ['residential', 'primary', 'secondary', 'tertiary', 'unclassified', 'living_street', 'trunk']

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
assert (DAY_DIR / 'day21.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'
RAW_DIR = DAY_DIR / 'data' / 'raw'
CACHE = DAY_DIR / 'data' / 'processed'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
PBF, EXTRACT_DATE = latest_pbf(REPO / 'reference' / 'raw' / 'osm')
CROSSWALKS_CSV = RAW_DIR / f'osm_crosswalks_{EXTRACT_DATE.isoformat()}.csv.gz'
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]
CA_GEOJSON = REPO / 'days' / 'day05-sight' / 'data' / 'raw' / 'community_areas_2026-11-04.geojson'
STATIONS_JSON = REPO / 'days' / 'day08-utopia' / 'output' / 'plate.json'  # ADA flags with Racine, Austin fixed
# Chicago's nine sides, by community area (as Plates X, XVII, XIX and XX): Wikipedia's grouping (unofficial)
SIDES = {
    'Central': [8, 32, 33],
    'North Side': [5, 6, 7, 21, 22],
    'Far North Side': [1, 2, 3, 4, 9, 10, 11, 12, 13, 14, 76, 77],
    'Northwest Side': [15, 16, 17, 18, 19, 20],
    'West Side': [23, 24, 25, 26, 27, 28, 29, 30, 31],
    'South Side': [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 60, 68, 69],
    'Southwest Side': [56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67],
    'Far Southwest Side': [70, 71, 72, 73, 74, 75],
    'Far Southeast Side': [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55],
}
SIDE_OF = {ca: s for s, cas in SIDES.items() for ca in cas}
assert sorted(SIDE_OF) == list(range(1, 78))
print(PBF.name, EXTRACT_DATE)

## 2. Pull
The city plus 200 m is cut from the Illinois extract (`osmium extract`, complete ways), then filtered to
crosswalk and sidewalk ways, streets and curb nodes (`osmium tags-filter`, which keeps the ways' nodes)
and written as OPL, a plain text line per object, which is read below.

In [ ]:
def osmium(*args):
    subprocess.run(['osmium', *map(str, args)], check=True)


opl = CACHE / f'chicago_ped_{EXTRACT_DATE.isoformat()}.opl'
if REFRESH or not opl.exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    area = gpd.read_file(BOUNDARY_RAW).to_crs(26916)
    area = gpd.GeoSeries([area.union_all().buffer(200)], crs=26916).to_crs(4326)
    poly = CACHE / 'area.geojson'
    poly.write_text(area.to_json())
    cut, ped = CACHE / 'chicago.osm.pbf', CACHE / 'ped.osm.pbf'
    osmium('extract', '-p', poly, '-s', 'complete_ways', '--overwrite', PBF, '-o', cut)
    osmium('tags-filter', '--overwrite', cut, 'w/footway=sidewalk,crossing', f'w/highway={",".join(STREETS)}',
           'n/barrier=kerb', 'n/kerb', '-o', ped)
    osmium('cat', '--overwrite', ped, '-f', 'opl', '-o', opl)
    for p in (poly, cut, ped):
        p.unlink()

ESC = re.compile(r'%([0-9a-f]+)%')
unesc = lambda s: ESC.sub(lambda m: chr(int(m.group(1), 16)), s)


def read_opl(path: Path):
    """Nodes {id: (lon, lat, tags)} and ways [(id, version, timestamp, tags, node ids)]."""
    nodes, ways = {}, []
    with open(path) as fh:
        for line in fh:
            f = line.rstrip('\n').split(' ')
            a = {p[0]: p[1:] for p in f[1:]}
            tags = {}
            for kv in a.get('T', '').split(','):
                if kv:
                    k, v = kv.split('=', 1)
                    tags[unesc(k)] = unesc(v)
            if f[0][0] == 'n':
                nodes[int(f[0][1:])] = (float(a['x']), float(a['y']), tags)
            elif f[0][0] == 'w':
                refs = [int(r[1:]) for r in a['N'].split(',')] if a.get('N') else []
                ways.append((int(f[0][1:]), int(a['v']), a['t'], tags, refs))
    return nodes, ways


nodes, ways = read_opl(opl)
print(f'{len(nodes):,} nodes, {len(ways):,} ways')

## 3. Validate

In [ ]:
curb = {i: t for i, (_, _, t) in nodes.items() if t.get('barrier') == 'kerb' or 'kerb' in t}
kind = lambda t: t.get('footway') if t.get('footway') in ('sidewalk', 'crossing') else ('street' if t.get('highway') in STREETS else None)
ways_df = pd.DataFrame([(i, v, ts, kind(t), refs) for i, v, ts, t, refs in ways], columns=['id', 'v', 'ts', 'kind', 'refs'])
assert ways_df['id'].is_unique
assert ways_df['kind'].notna().all()
kerb_values = pd.Series([t.get('kerb', '(none)') for t in curb.values()]).value_counts()
assert set(KNOWN) <= set(kerb_values.index), 'An expected kerb value is missing'
print(ways_df['kind'].value_counts().to_dict())
print(f'{len(curb):,} curb nodes')
kerb_values

## 4. Transform
Coordinates in miles from State & Madison (`atlas.grid`, accurate to well under 1% across the city).
A crosswalk is in the city if its midpoint is. Each end takes the curb node among its first (or last) two
nodes, the one nearer the end first; its value is the `kerb` tag, `yes` for a bare `barrier=kerb`, or
empty when no curb is mapped.

In [ ]:
def end_value(refs):
    for r in refs:
        if r in curb:
            return curb[r].get('kerb', 'yes')
    return ''


cw = ways_df[ways_df['kind'] == 'crossing'].copy()
cw = cw[cw['refs'].map(lambda r: len(r) >= 2 and all(x in nodes for x in r))]
cw['end0'] = cw['refs'].map(lambda r: end_value(r[:2]))
cw['end1'] = cw['refs'].map(lambda r: end_value(r[::-1][:2]))
first, last = cw['refs'].map(lambda r: nodes[r[0]]), cw['refs'].map(lambda r: nodes[r[-1]])
cw['lon0'], cw['lat0'] = first.map(lambda n: n[0]), first.map(lambda n: n[1])
cw['lon1'], cw['lat1'] = last.map(lambda n: n[0]), last.map(lambda n: n[1])

if REFRESH or not CROSSWALKS_CSV.exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    cols = ['id', 'v', 'lat0', 'lon0', 'lat1', 'lon1', 'end0', 'end1']
    cw[cols].round({'lat0': 7, 'lon0': 7, 'lat1': 7, 'lon1': 7}).to_csv(CROSSWALKS_CSV, index=False)
cx = pd.read_csv(CROSSWALKS_CSV, keep_default_na=False)
assert len(cx) == len(cw)

# miles from State & Madison; the city and its community areas in the same miles
cx['x0'], cx['y0'] = miles_from_origin(cx['lat0'], cx['lon0'])
cx['x1'], cx['y1'] = miles_from_origin(cx['lat1'], cx['lon1'])
cx['x'], cx['y'] = (cx['x0'] + cx['x1']) / 2, (cx['y0'] + cx['y1']) / 2


def to_miles(geom):
    return shapely.transform(geom, lambda c: np.column_stack(miles_from_origin(c[:, 1], c[:, 0])))


city = to_miles(gpd.read_file(BOUNDARY_RAW).union_all())
shapely.prepare(city)
cx = cx[shapely.contains_xy(city, cx['x'], cx['y'])].reset_index(drop=True)
cx['known'] = cx['end0'].isin(KNOWN).astype(int) + cx['end1'].isin(KNOWN).astype(int)

ca = gpd.read_file(CA_GEOJSON)
ca['num'] = ca['area_numbe'].astype(int)
ca['geometry'] = [to_miles(g) for g in ca.geometry]
ca_tree = shapely.STRtree(ca.geometry.values)
hit = ca_tree.query(shapely.points(cx['x'], cx['y']), predicate='within')
cx['ca'] = 0
cx.loc[hit[0], 'ca'] = ca['num'].to_numpy()[hit[1]]
cx['side'] = cx['ca'].map(SIDE_OF)
print(f'{len(cx):,} crosswalks in the city; {(cx["ca"] == 0).sum()} outside every community area (edge)')
cx['known'].value_counts().sort_index()

In [ ]:
# 'L' stations in the city, with the ADA flags as Plate VIII corrected them (Racine, Austin)
doc8 = json.loads(STATIONS_JSON.read_text())
def y_miles(y):
    y = np.asarray(y, float)
    return np.where(y >= 0, y / 800, np.where(y >= -1200, y / 1200, -1 + (y + 1200) / 800))
st = pd.DataFrame([s for s in doc8['stations'] if s['id'].isdigit() and s['inCity']])
st['mx'], st['my'] = st['x'] / 800, y_miles(st['y'])
xy = np.column_stack([cx['x'], cx['y']])
def around(sx, sy, r=QUARTER):
    return np.hypot(xy[:, 0] - sx, xy[:, 1] - sy) <= r
rows = []
for s in st.itertuples():
    m = around(s.mx, s.my)
    d = cx[m]
    rows.append({'n': int(m.sum()), 'both': int((d['known'] == 2).sum()), 'one': int((d['known'] == 1).sum()),
                 'none': int((d['known'] == 0).sum()),
                 'ends': int((~d['end0'].isin(KNOWN)).sum() + (~d['end1'].isin(KNOWN)).sum())})
st = pd.concat([st.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
st['share'] = st['both'] / st['n'].clip(lower=1)
print(f'{len(st)} city stations, {st["ada"].sum()} accessible')
st[['name', 'ada', 'n', 'both', 'share']].sort_values('share').head()

## 5. Analyze

In [ ]:
N = len(cx)
counts = cx['known'].value_counts().reindex([2, 1, 0], fill_value=0)
ends = pd.concat([cx['end0'], cx['end1']]).replace('', '(none)')
given = ends[ends.isin(KNOWN)]
acc_zone = np.zeros(N, bool)
for s in st[st['ada']].itertuples():
    acc_zone |= around(s.mx, s.my)
acc = cx[acc_zone]
acc_st = st[st['ada']]
sides = []
for s in SIDES:
    d = cx[cx['side'] == s]
    sides.append({'side': s, 'n': len(d), 'both': int((d['known'] == 2).sum()), 'one': int((d['known'] == 1).sum()),
                  'none': int((d['known'] == 0).sum())})
sides.sort(key=lambda r: -r['both'] / r['n'])
summary = {
    'n': N, 'both': int(counts[2]), 'one': int(counts[1]), 'none': int(counts[0]),
    'ends': int(len(ends)), 'endsGiven': int(len(given)), 'endsCurbNoHeight': int(ends.isin(['yes', 'unspecified']).sum()),
    'endsNoCurb': int((ends == '(none)').sum()),
    'endValues': {k: int(v) for k, v in ends.value_counts().items()},
    'givenPassable': float(given.isin(PASSABLE).mean()), 'raised': int((given == 'raised').sum()),
    'accStations': int(len(acc_st)), 'accN': int(len(acc)), 'accBoth': int((acc['known'] == 2).sum()),
    'accOne': int((acc['known'] == 1).sum()), 'accNone': int((acc['known'] == 0).sum()),
    'accEndsToTag': int((~acc['end0'].isin(KNOWN)).sum() + (~acc['end1'].isin(KNOWN)).sum()),
    'accZero': int((acc_st['both'] == 0).sum()), 'accHalf': int((acc_st['share'] >= 0.5).sum()),
    'stations': int(len(st)), 'stationMedian': float(st['share'].median()),
}
print(json.dumps(summary, indent=1))
pd.DataFrame(sides).assign(pct=lambda d: (100 * d['both'] / d['n']).round(1))

In [ ]:
# The inset: the accessible stations with the most and the fewest crosswalks giving both heights in their
# quarter mile (at least MIN_CROSSWALKS crosswalks); ties go to the station with more crosswalks
cand = st[st['ada'] & (st['n'] >= MIN_CROSSWALKS)].sort_values(['share', 'n'], ascending=[False, False])
best, worst = cand.iloc[0], cand.sort_values(['share', 'n'], ascending=[True, False]).iloc[0]
print(best['name'], f'{best.share:.0%}', '|', worst['name'], f'{worst.share:.0%}')

# lines for the inset windows: sidewalks drawn in pale, crosswalks with their ends
line_of = lambda refs: shapely.LineString(np.column_stack(miles_from_origin([nodes[r][1] for r in refs], [nodes[r][0] for r in refs])))
side_ways = ways_df[(ways_df['kind'] == 'sidewalk') & ways_df['refs'].map(lambda r: len(r) >= 2 and all(x in nodes for x in r))]
side_lines = shapely.STRtree([line_of(r) for r in side_ways['refs']])
side_geoms = side_lines.geometries
def window(s, r=QUARTER, ry=INSET_RY):
    box = shapely.box(s.mx - r, s.my - ry, s.mx + r, s.my + ry)
    walks = [shapely.clip_by_rect(side_geoms[i], *box.bounds) for i in side_lines.query(box)]
    walks = [g for g in walks if not g.is_empty]
    segs = []
    for g in walks:
        for part in getattr(g, 'geoms', [g]):
            segs.append([[round(float(x - s.mx), 4), round(float(y - s.my), 4)] for x, y in part.coords])
    inside = (np.abs(cx['x'] - s.mx) <= r + 0.02) & (np.abs(cx['y'] - s.my) <= ry + 0.02)
    cws = [[round(float(c.x0 - s.mx), 4), round(float(c.y0 - s.my), 4), round(float(c.x1 - s.mx), 4), round(float(c.y1 - s.my), 4), c.end0, c.end1]
           for c in cx[inside].itertuples()]
    others = [[str(o.id), round(float(o.mx - s.mx), 4), round(float(o.my - s.my), 4), bool(o.ada)] for o in st.itertuples()
              if o.id != s.id and abs(o.mx - s.mx) <= r and abs(o.my - s.my) <= ry]
    return {'id': str(s.id), 'r': r, 'ry': ry, 'sidewalks': segs, 'crosswalks': cws, 'others': others}
inset = [window(best), window(worst)]
[(w['id'], len(w['sidewalks']), len(w['crosswalks'])) for w in inset]

## 6. QA

In [ ]:
qa = {}
# Curb nodes off the crosswalk: ends with no curb node of their own but one within 5 m (miles: 5 / 1609.344)
curb_xy = np.column_stack(miles_from_origin([nodes[i][1] for i in curb], [nodes[i][0] for i in curb]))
tree = shapely.STRtree(shapely.points(curb_xy))
bare = np.concatenate([cx.loc[cx['end0'] == '', ['x0', 'y0']].to_numpy(), cx.loc[cx['end1'] == '', ['x1', 'y1']].to_numpy()])
idx, dist = tree.query_nearest(shapely.points(bare), max_distance=5 / 1609.344, return_distance=True)
qa['endsNoCurb'] = int(len(bare))
qa['endsCurbWithin5m'] = int(len(np.unique(idx[0])))
# The loose reading: any curb node at an end counts, with or without a height
loose = (cx['end0'] != '').astype(int) + (cx['end1'] != '').astype(int)
qa['looseBoth'] = float((loose == 2).mean())
qa['looseNone'] = float((loose == 0).mean())
# Streets with a sidewalk drawn beside them: the midpoint of each street way within 30 m of a sidewalk line
streets = ways_df[(ways_df['kind'] == 'street') & ways_df['refs'].map(lambda r: len(r) >= 2 and all(x in nodes for x in r))]
sl = [line_of(r) for r in streets['refs']]
mid = shapely.line_interpolate_point(sl, 0.5, normalized=True)
inside = shapely.contains_xy(city, shapely.get_x(mid), shapely.get_y(mid))
length = shapely.length(sl)
near = np.zeros(len(sl), bool)
qi, _ = side_lines.query_nearest(mid, max_distance=SIDEWALK_NEAR_M / 1609.344, return_distance=True)
near[qi[0]] = True
qa['streetMiles'] = float(length[inside].sum())
qa['streetShareWithSidewalk'] = float(length[inside & near].sum() / length[inside].sum())
# When the lines were drawn: OSM ids rise with time, so each way's creation date is read off the
# version-1 ways (whose timestamp is their creation) by id
v1 = ways_df[ways_df['v'] == 1].sort_values('id')
t1 = np.maximum.accumulate(pd.to_datetime(v1['ts'], utc=True).dt.tz_convert(None).astype('datetime64[s]').astype('int64').to_numpy())
created = lambda ids: pd.to_datetime(np.interp(ids, v1['id'], t1).astype('int64'), unit='s', utc=True)
sw_len = shapely.length([line_of(r) for r in side_ways['refs']])
sw_new = created(side_ways['id']) >= pd.Timestamp('2022-01-01', tz='UTC')
qa['sidewalkMiles'] = float(sw_len.sum())
qa['sidewalkSince2022'] = float(sw_len[np.asarray(sw_new)].sum() / sw_len.sum())
qa['crosswalksSince2022'] = float((created(cx['id']) >= pd.Timestamp('2022-01-01', tz='UTC')).mean())
assert summary['both'] + summary['one'] + summary['none'] == N
assert sum(r['n'] for r in sides) == (cx['ca'] > 0).sum()
qa

## 7. Export

In [ ]:
pct = lambda v: f'{round(100 * v)}%'
finding = (f'OpenStreetMap draws {N:,} crosswalks in Chicago. At {pct(summary["none"] / N)} of them it does not say '
           f'how high the curb is at either end.')
ledger = Ledger(
    source='OpenStreetMap contributors, Geofabrik extract of Illinois',
    dataset_id=PBF.name, url=OSM_URL, pulled=EXTRACT_DATE,
    units='crosswalks (footway=crossing ways)',
    method=['Crosswalk ways inside the city limits (midpoint)',
            'An end gives the height if its curb node (first or last two nodes) has kerb=lowered, flush, raised or no',
            'Stations: city L stations, ADA flags as Plate VIII; straight-line quarter mile'],
    caveats=['The map, not the street: a curb without a height may well be lowered',
             'Crossings drawn only as points (alleys, driveways) are not counted'],
)
payload = {
    'extract': {'file': PBF.name, 'date': EXTRACT_DATE.isoformat(), 'url': GEOFABRIK_URL},
    'sources': {'osm': OSM_URL, 'kerb': KERB_WIKI, 'crossing': CROSSING_WIKI},
    'known': KNOWN, 'quarter': QUARTER,
    'summary': summary, 'qa': qa, 'sides': sides,
    'stationCols': ['id', 'short', 'name', 'lines', 'ada', 'x', 'y', 'n', 'both', 'one', 'none', 'ends'],
    'stations': [[str(s.id), s.short, s.name, s.lines, bool(s.ada), round(float(s.mx), 4), round(float(s.my), 4),
                  int(s.n), int(s.both), int(s.one), int(s.none), int(s.ends)] for s in st.itertuples()],
    'inset': inset,
    # every crosswalk: midpoint in miles and how many ends give the height
    'cross': [[round(float(x), 3), round(float(y), 3), int(k)] for x, y, k in zip(cx['x'], cx['y'], cx['known'])],
}
write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=finding, payload=payload)
log.info('Wrote %s (%.0f KB)', OUT_PATH, OUT_PATH.stat().st_size / 1024)
finding